# Vanishing & Exploding Gradients: Spectral Analysis

Companion notebook for the [Vanishing Gradients wiki page](https://ml-viz-ruby.vercel.app/wiki/vanishing-gradient-analysis).

We plot the spectral-norm bound $\lambda^n$ for several spectral radii,
empirically verify the decay/growth tables, and show how LSTM gating
maintains a near-unit per-step Jacobian.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — The spectral norm bound

The gradient magnitude over a gap of $n = T-k$ steps is bounded by:
$$\left\|\frac{\partial \mathcal{L}_T}{\partial \mathbf{h}_k}\right\| \le C \cdot \lambda^n$$
where $\lambda = \|W_{hh}\|$ is the spectral norm (largest singular value) of $W_{hh}$.

In [ ]:
gaps = np.arange(1, 55)

radii = [
    (0.5,  '#6366f1', 'λ=0.5  (vanishing)'),
    (0.8,  '#20d9d2', 'λ=0.8  (slow decay)'),
    (1.0,  '#888',    'λ=1.0  (neutral)'),
    (1.2,  '#f97316', 'λ=1.2  (slow explode)'),
    (1.5,  '#ef4444', 'λ=1.5  (fast explode)'),
]

fig, ax = plt.subplots(figsize=(9, 5))
for lam, color, label in radii:
    bound = lam ** gaps
    ax.semilogy(gaps, bound, color=color, label=label)

ax.axhline(1.0, color='#555', linestyle='--', linewidth=1)
ax.set_xlabel('Gap n = T − k (number of steps back)')
ax.set_ylabel('Gradient bound λⁿ (log scale)')
ax.set_title('Spectral norm bound: λⁿ vs. gap')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2 — Numeric tables from the wiki

Reproduce the exact values from the wiki's worked example.

In [ ]:
import math

def tanh_prime(z):
    return 1 - math.tanh(z)**2

print(f"max tanh'(z) at z=0:   {tanh_prime(0.0):.4f}")
print(f"tanh'(1):               {tanh_prime(1.0):.4f}")
print(f"tanh'(2):               {tanh_prime(2.0):.4f}")
print()

print(f"{'Gap':>4}  {'λ=0.5':>12}  {'λ=1.0':>12}  {'λ=1.5':>12}")
print("-" * 50)
for n in [1, 5, 10, 20, 50]:
    print(f"{n:>4}  {0.5**n:>12.3e}  {1.0**n:>12.3e}  {1.5**n:>12.3e}")

print()
print("With tanh'≈0.4 (saturated), effective gain per step = 0.5×0.4 = 0.2:")
print(f"{'Gap':>4}  {'0.2^n':>12}")
for n in [5, 10, 20]:
    print(f"{n:>4}  {0.2**n:>12.3e}")

## 3 — Empirical gradient norms vs. theory

Run a random RNN and measure actual gradient norms at different depths.

In [ ]:
def empirical_grad_norm(T, W_hh_val, n_trials=200):
    """Average |dh_T/dh_1| over many random hidden-state trajectories."""
    norms = []
    for _ in range(n_trials):
        x = np.random.randn(T)
        h = [0.0]
        a_vals = []
        for t in range(T):
            a_t = W_hh_val * h[-1] + 0.5 * x[t]
            a_vals.append(a_t)
            h.append(float(np.tanh(a_t)))
        # Product of Jacobians from step T to step 1
        grad = 1.0
        for t in reversed(range(1, T)):
            grad *= (1 - np.tanh(a_vals[t])**2) * W_hh_val
        norms.append(abs(grad))
    return np.mean(norms)

T_range = [1, 3, 5, 10, 15, 20]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax_idx, (W_val, color, label) in enumerate([(0.8, '#6366f1', 'λ=0.8'), (1.2, '#f97316', 'λ=1.2')]):
    empirical = [empirical_grad_norm(T, W_val) for T in T_range]
    theory = [W_val**(T-1) for T in T_range]

    ax = axes[ax_idx]
    ax.semilogy(T_range, empirical, 'o-', color=color, label='Empirical mean')
    ax.semilogy(T_range, theory, '--', color='#888', label=f'Theory λ^(T-1)')
    ax.set_title(f'{label}: empirical vs. theory')
    ax.set_xlabel('Sequence length T')
    ax.set_ylabel('|dh_T/dh_1|')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4 — LSTM cell state: near-unit Jacobian

The LSTM cell state update is additive:
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

When the forget gate $f_t \approx 1$, the Jacobian $\partial c_t / \partial c_{t-1} \approx 1$,
bypassing the vanishing gradient problem.

In [ ]:
def lstm_cell_jacobian(forget_gate_val, T):
    """Product of per-step cell-state Jacobians for a constant forget gate."""
    return forget_gate_val ** T

T_vals = np.arange(1, 51)
fig, ax = plt.subplots(figsize=(9, 4))

configs = [
    (1.0, '#20d9d2', 'f=1.0 (open forget gate)'),
    (0.9, '#6366f1', 'f=0.9'),
    (0.7, '#f97316', 'f=0.7'),
    (0.5, '#ef4444', 'f=0.5 (like vanilla RNN)'),
]

for f_val, color, label in configs:
    ax.plot(T_vals, [lstm_cell_jacobian(f_val, T) for T in T_vals],
            color=color, label=label)

ax.set_xlabel('Gap T (steps)')
ax.set_ylabel('Cell-state Jacobian product')
ax.set_title('LSTM forget-gate controls the Jacobian (linear scale)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1 — Phase diagram

Create a 2D heatmap with $\lambda \in [0.5, 1.5]$ on the x-axis and gap $n \in [1, 30]$
on the y-axis, coloring each cell by $\log_{10}(\lambda^n)$. Where is the boundary
between vanishing and exploding?

In [ ]:
# TODO(you): create the heatmap
lambdas = np.linspace(0.5, 1.5, 50)
ns = np.arange(1, 31)

# LAM, N = np.meshgrid(lambdas, ns)
# log_bound = np.log10(LAM ** N)   # fill this in

# plt.figure(figsize=(9, 5))
# plt.pcolormesh(LAM, N, log_bound, cmap='RdBu_r', vmin=-15, vmax=15)
# plt.colorbar(label='log10(λⁿ)')
# plt.axvline(1.0, color='white', linestyle='--')
# plt.xlabel('Spectral norm λ')
# plt.ylabel('Gap n')
# plt.title('Vanishing (blue) vs. exploding (red)')
# plt.show()

### Exercise 2 — Gradient clipping

Simulate gradient descent on a simple scalar RNN with $\lambda=1.4$ and $T=20$.
Run with and without gradient clipping (threshold $c=1$) and plot the gradient norms
over training steps. Show that clipping stabilizes training but doesn't fix the
vanishing case when $\lambda=0.6$.

<details>
<summary>Solution outline</summary>

```python
def clip_grad(g, c=1.0):
    norm = abs(g)
    return g * (c / norm) if norm > c else g

# For λ=1.4: without clipping the gradient blows up; with clipping it stays bounded.
# For λ=0.6: the gradient is already near zero — clipping has no effect on learning.
```
</details>